In [ ]:
import torch
import numpy as np
import random
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # For deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(0)

In [ ]:
# =========================================================================================
# Load Data
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
import torch
from torch.utils.data import TensorDataset, DataLoader

# Read the waveforms from the CSV file "B_waveform[T].csv"
waveforms = np.loadtxt("N87/B_waveform[T].csv", delimiter=",")
num_waveforms, seq_len = waveforms.shape
print(f"Number of waveforms: {num_waveforms}")
print(f"Sequence length: {seq_len}")

num_harmonics = 13

# Load additional input variables
frequencies = np.loadtxt("N87/Frequency[Hz].csv", delimiter=",")
temperatures = np.loadtxt("N87/Temperature[C].csv", delimiter=",")

if frequencies.ndim == 1:
    frequencies = frequencies.reshape(-1, 1)
if temperatures.ndim == 1:
    temperatures = temperatures.reshape(-1, 1)

# Compute harmonic magnitudes
harmonic_magnitudes = np.zeros((waveforms.shape[0], num_harmonics))
for i in range(waveforms.shape[0]):
    waveform = waveforms[i]
    freq = frequencies[i, 0]
    n = len(waveform)
    fft_result = np.fft.rfft(waveform)
    fft_magnitude = np.abs(fft_result) / n * 2
    sample_rate = freq * n
    freqs = np.fft.rfftfreq(n, d=1.0/(freq * n))
    for k in range(1, num_harmonics + 1):
        harmonic_freq = k * freq
        idx = np.argmin(np.abs(freqs - harmonic_freq))
        harmonic_magnitudes[i, k-1] = fft_magnitude[idx]

# Load output prediction variable
volumetric_losses = np.loadtxt("N87/Volumetric_losses[Wm-3].csv", delimiter=",")
if volumetric_losses.ndim == 1:
    volumetric_losses = volumetric_losses.reshape(-1, 1)

# Prepare input features
X_full = np.hstack([harmonic_magnitudes, frequencies, temperatures])
y_full = volumetric_losses

# Downsample to 4071 with stratification to preserve the distribution of y
epsilon = 1e-12
y_log_full = np.log(y_full.flatten() + epsilon)
num_bins = 20
y_bins = np.digitize(y_log_full, np.histogram(y_log_full, bins=num_bins)[1][1:-1])

sss = StratifiedShuffleSplit(n_splits=1, test_size=(waveforms.shape[0]-1000)/waveforms.shape[0], random_state=42)
for down_idx, _ in sss.split(X_full, y_bins):
    X = X_full[down_idx]
    y = y_full[down_idx]
    y_log = y_log_full[down_idx]

print(f"Full input feature matrix X shape: {X_full.shape}")
print(f"Full output variable y shape: {y_full.shape}")
print(f"Downsampled input feature matrix X shape: {X.shape}")
print(f"Downsampled output variable y shape: {y.shape}")

In [ ]:
# =========================================================================================
# Define LoRA Linear Layer
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """
    LoRA-enhanced Linear layer.
    The layer computes: output = W @ x + scale * B @ A @ x
    where W is the frozen pretrained weight, and B @ A is the low-rank update.
    """
    def __init__(self, in_features, out_features, rank=8, lora_alpha=16, lora_dropout=0.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout
        
        # Trainable LoRA parameters
        self.lora_a = nn.Parameter(torch.zeros(rank, in_features))
        self.lora_b = nn.Parameter(torch.zeros(out_features, rank))
        self.scale = lora_alpha / rank
        
        # Initialization
        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))
        nn.init.zeros_(self.lora_b)
        
        if lora_dropout > 0:
            self.dropout = nn.Dropout(lora_dropout)
        else:
            self.dropout = nn.Identity()

    def forward(self, x):
        # x: (batch_size, in_features)
        # Apply LoRA: scale * (B @ A) @ x
        lora_out = torch.matmul(
            torch.matmul(x, self.lora_a.t()),  # x @ A^T = (batch_size, rank)
            self.lora_b.t()  # B^T = (rank, out_features)
        )  # (x @ A^T) @ B^T = (batch_size, out_features)
        return self.scale * lora_out


class FeedforwardNNWithLoRA(nn.Module):
    """
    FNN with LoRA layers.
    Combines original frozen weights with trainable LoRA updates.
    """
    def __init__(self, pretrained_model, rank=8, lora_alpha=16, lora_dropout=0.0):
        super().__init__()
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout
        
        # Store pretrained model layers and create LoRA layers
        self.lora_layers = nn.ModuleDict()
        self.pretrained_layers = nn.ModuleDict()
        
        # Extract Linear layers from the pretrained model's sequential
        layer_idx = 0
        for i, module in enumerate(pretrained_model.net.modules()):
            if isinstance(module, nn.Linear):
                # Freeze pretrained weights
                module.weight.requires_grad = False
                module.bias.requires_grad = False
                
                self.pretrained_layers[f"linear_{layer_idx}"] = module
                
                # Create LoRA layer
                self.lora_layers[f"lora_{layer_idx}"] = LoRALinear(
                    module.in_features,
                    module.out_features,
                    rank=rank,
                    lora_alpha=lora_alpha,
                    lora_dropout=lora_dropout
                )
                layer_idx += 1
        
        # Store activation functions
        self.activation = nn.SiLU() # to be adjusted
        self.num_lora_layers = layer_idx
        
        # Recreate the sequential model structure
        self._build_net(pretrained_model)
    
    def _build_net(self, pretrained_model):
        """Rebuild the network with LoRA layers integrated"""
        layers = []
        lora_idx = 0
        
        for module in pretrained_model.net.modules():
            if isinstance(module, nn.Linear):
                layers.append(self.pretrained_layers[f"linear_{lora_idx}"])
                lora_idx += 1
            elif isinstance(module, nn.SiLU):
                layers.append(nn.SiLU())
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        # Forward pass through the base network
        base_out = self.net(x)
        
        # Add LoRA contributions layer by layer
        current = x
        lora_contribution = 0
        
        layer_idx = 0
        for i, module in enumerate(self.pretrained_layers.values()):
            if layer_idx == 0:
                # First layer
                lora_update = self.lora_layers[f"lora_{layer_idx}"](x)
                lora_contribution = lora_update
                # Get intermediate for next layer
                current = self.activation(module(x) + lora_update)
            elif layer_idx < self.num_lora_layers - 1:
                # Hidden layers
                lora_update = self.lora_layers[f"lora_{layer_idx}"](current)
                lora_contribution = lora_contribution + lora_update
                current = self.activation(module(current) + lora_update)
            else:
                # Output layer
                base_out = module(current)
                lora_out = self.lora_layers[f"lora_{layer_idx}"](current)
                return base_out + lora_out
                # lora_update = self.lora_layers[f"lora_{layer_idx}"](current)
                # lora_contribution = lora_contribution + lora_update
            
            layer_idx += 1
        
        # Return base output + LoRA contribution
        # return base_out + lora_contribution


print("LoRA model classes defined successfully!")

In [ ]:
# =========================================================================================
# Define Original Model Architecture

class FeedforwardNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=1, num_hidden_layers=2):
        super().__init__()
        layers = []
        # Input layer
        layers.append(nn.Linear(in_dim, hidden_dim))
        layers.append(nn.SiLU())
        # Hidden layers
        for _ in range(num_hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.SiLU())
        # Output layer
        layers.append(nn.Linear(hidden_dim, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

print("Original model architecture defined!")

In [ ]:
# ===============================
# Fixed Test Set (20%)
X_avail, X_test, y_avail, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scaler (fit ONLY on training data)
input_scaler = RobustScaler()
output_scaler = StandardScaler()


In [ ]:
# =========================================================================================
# Load Pretrained Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create and load the original pretrained model
in_dim = X.shape[1]  # 15 features
hidden_dim = 256
out_dim = 1
num_hidden_layers = 5

pretrained_model = FeedforwardNN(in_dim, hidden_dim, out_dim, num_hidden_layers).to(device)
pretrained_model.load_state_dict(torch.load("models/pretrained_3C90.pth", map_location=device))
pretrained_model.eval()
print("Pretrained model loaded successfully!")

# Count parameters in pretrained model
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

pretrained_params = count_parameters(pretrained_model)
print(f"Pretrained model parameters: {pretrained_params}")

In [ ]:
# =========================================================================================
# Fine-Tuning Hyperparameters
num_epochs = 500
batch_size = 32

lora_rank = 16  # Rank of LoRA matrices
lora_alpha = 16  # Scaling factor
lora_dropout = 0.1  # Dropout in LoRA layers

In [ ]:
# =========================================================================================
# LoRA Fine-Tuning
def train_lora(pretrained_model, train_loader, val_loader):
    model = FeedforwardNNWithLoRA(
        pretrained_model, lora_rank, lora_alpha, lora_dropout
    ).to(device)

    lora_params_list = list(model.lora_layers.parameters())
    optimizer = torch.optim.Adam(lora_params_list, lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_model_state = None
    patience_cnt = 0
    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        scheduler.step()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= 100:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model

In [ ]:
# =========================================================================================
# Full Fine-Tuning
def train_full_finetune(pretrained_model, train_loader, val_loader):
    model = copy.deepcopy(pretrained_model).to(device)

    for p in model.parameters():
        p.requires_grad = True
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_model_state = None
    patience_cnt = 0
    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        scheduler.step()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= 100:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model

In [ ]:
# =========================================================================================
# Partial Fine-Tuning
def train_partial_finetune(pretrained_model, train_loader, val_loader):
    model = copy.deepcopy(pretrained_model).to(device)

    layers = list(model.net.children())

    num_unfreeze = 5
    for i, layer in enumerate(layers):
        if i < (len(layers) - num_unfreeze):
            for param in layer.parameters():
                param.requires_grad = False
        else:
            for param in layer.parameters():
                param.requires_grad = True
            
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_model_state = None
    patience_cnt = 0
    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        scheduler.step()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= 100:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model

In [ ]:
# =========================================================================================
# Training from Scratch
def train_from_scratch(train_loader, val_loader):
    model = FeedforwardNN(in_dim, hidden_dim, out_dim, num_hidden_layers).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_model_state = None
    patience_cnt = 0
    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        scheduler.step()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= 100:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model

In [ ]:
# =========================================================================================
# Evaluate Model Accuracy

def evaluate_model_accuracy(loader, scaler_y, model, device, set_name="Set"):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            preds = model(xb).cpu().numpy()
            y_pred.append(preds)
            y_true.append(yb.cpu().numpy())
    y_true = np.vstack(y_true)
    y_pred = np.vstack(y_pred)
    # Inverse transform to original y scale
    y_true_log = scaler_y.inverse_transform(y_true)
    y_pred_log = scaler_y.inverse_transform(y_pred)
    y_true_orig = np.exp(y_true_log)
    y_pred_orig = np.exp(y_pred_log)
    mae = np.mean(np.abs(y_true_orig - y_pred_orig))
    rmse = np.sqrt(np.mean((y_true_orig - y_pred_orig) ** 2))
    epsilon = 1e-8
    mape = np.mean(np.abs((y_true_orig - y_pred_orig) / (y_true_orig + epsilon))) * 100
    size = y_true_orig.shape[0]
    return mae, rmse, mape, size, y_true_orig, y_pred_orig

In [ ]:
# =========================================================================================
# Experiment Loop
import pandas as pd
import copy
ratios = [0.05, 0.10, 0.20, 0.50]
results = []

# Utility: Subsample target data
def subsample_training_data(X, y, ratio, seed=42):
    if ratio == 1.0:
        return X, y
    else:
        X_sub, _, y_sub, _ = train_test_split(X, y, train_size=ratio, random_state=seed)
        return X_sub, y_sub

for r in ratios:
    print(f"\n===== Target data ratio: {int(r*100)}% =====")

    X_small, y_small = subsample_training_data(X_avail, y_avail, r)
    X_train, X_val, y_train, y_val = train_test_split(
        X_small, y_small, test_size=0.2, random_state=42
    )

    X_train_scaled = input_scaler.fit_transform(X_train)
    X_val_scaled = input_scaler.transform(X_val)
    X_test_scaled = input_scaler.transform(X_test)

    y_train_log = np.log(y_train + epsilon)
    y_val_log = np.log(y_val + epsilon)
    y_test_log = np.log(y_test + epsilon)
    
    y_train_scaled = output_scaler.fit_transform(y_train_log.reshape(-1, 1))
    y_val_scaled = output_scaler.transform(y_val_log.reshape(-1, 1))
    y_test_scaled = output_scaler.transform(y_test_log.reshape(-1, 1))

    train_set = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), torch.tensor(y_train_scaled, dtype=torch.float32))
    val_set = TensorDataset(torch.tensor(X_val_scaled, dtype=torch.float32), torch.tensor(y_val_scaled, dtype=torch.float32))
    test_set = TensorDataset(torch.tensor(X_test_scaled, dtype=torch.float32), torch.tensor(y_test_scaled, dtype=torch.float32))
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    # LoRA
    lora_model = train_lora(pretrained_model, train_loader, val_loader)
    mae, rmse, mape, _, _, _ = evaluate_model_accuracy(test_loader, output_scaler, lora_model, device, set_name="LoRA Set")
    results.append({"Ratio": r, "Method": "LoRA", "MAE": mae, "RMSE": rmse, "MAPE": mape})

    # Full FT
    ft_model = train_full_finetune(pretrained_model, train_loader, val_loader)
    mae, rmse, mape, _, _, _ = evaluate_model_accuracy(test_loader, output_scaler, ft_model, device, set_name="Full FT Set")
    results.append({"Ratio": r, "Method": "Full FT", "MAE": mae, "RMSE": rmse, "MAPE": mape})

    # Partial FT
    ft_model = train_partial_finetune(pretrained_model, train_loader, val_loader)
    mae, rmse, mape, _, _, _ = evaluate_model_accuracy(test_loader, output_scaler, ft_model, device, set_name="Partial FT Set")
    results.append({"Ratio": r, "Method": "Partial FT", "MAE": mae, "RMSE": rmse, "MAPE": mape})

    # Scratch
    sc_model = train_from_scratch(train_loader, val_loader)
    mae, rmse, mape, _, _, _ = evaluate_model_accuracy(test_loader, output_scaler, sc_model, device, set_name="Scratch Set")
    results.append({"Ratio": r, "Method": "Scratch", "MAE": mae, "RMSE": rmse, "MAPE": mape})

# Save results
df = pd.DataFrame(results)
df.to_csv("0-100_3C90-N87.csv", index=False)
print(df)